In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    IntervalStrategy,
    Trainer,
    TrainingArguments,
    pipeline
)
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.utils import resample
from accelerate import notebook_launcher
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig, PeftConfig
import pandas as pd
import torch

In [2]:
# Finnish FinBERT sentiment model (TurkuNLP)
MODEL_NAME = "TurkuNLP/finnish-modernbert-large"

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
finnbert_huggingface_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# Set device: MPS if available (Apple Silicon), else CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at TurkuNLP/finnish-modernbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# Create pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=finnbert_huggingface_model,
    tokenizer=tokenizer,
    device=device
)

Device set to use mps


In [4]:
cleaned_sentment_dataset = pd.read_pickle('../data/finnsentiment_cleaned_unambiguous.pkl')

# Replace with your dataset loading
ds = cleaned_sentment_dataset

In [5]:
ds.head(10)

,label,text
2,1,40.
3,2,Kyseessä voi olla loppuelämäsi nainen.
4,2,Sinne vaan ocean clubiin iskemään!
5,2,Itsekin pidän Keskustan kampanjointia ihan hyv...
6,1,"Kamppi, Kontula, Kluuvi"
8,0,en haluaisi että kissani vuotaa.. =)
9,0,Nyt olisi lääkitys paikallaan.
10,1,Ihmiset joutuvat joskus monenlaisien päätelmie...
11,0,"Eniten pelkään sitä, että jos mies vain koko a..."
12,2,Muutenkin suosittelen kaikille asiasta kiinnos...


In [6]:
ds["label"].value_counts()

label
1    12374
2     1358
0     1230
Name: count, dtype: int64

In [7]:
df_0 = ds[ds['label'] == 0]
df_1 = ds[ds['label'] == 1]
df_2 = ds[ds['label'] == 2]

# Undersample class 1
df_1_balanced = df_1.sample(n=min(len(df_0), len(df_2)), random_state=42)

# Combine for balanced dataset
balanced_ds = pd.concat([df_0, df_1_balanced, df_2])

# balanced_ds = ds

In [8]:
# Upsample classes 0 and 2
# df_0_upsampled = resample(df_0, replace=True, n_samples=len(df_1), random_state=42)
# df_2_upsampled = resample(df_2, replace=True, n_samples=len(df_1), random_state=42)

# balanced_ds = pd.concat([df_1, df_0_upsampled, df_2_upsampled])

In [9]:
balanced_ds["label"].value_counts()

label
2    1358
0    1230
1    1230
Name: count, dtype: int64

In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Split your DataFrame
train_df, val_df = train_test_split(balanced_ds, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Preprocess using map (not pandas apply)
def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
    enc["label"] = batch["label"]
    return enc

train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset = val_dataset.map(preprocess, batched=True)

# Create DatasetDict for Trainer
ds_preprocessed = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(axis=-1)
    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    recall = recall_score(y_true=labels, y_pred=pred, average="weighted")
    precision = precision_score(y_true=labels, y_pred=pred, average="weighted")
    f1 = f1_score(y_true=labels, y_pred=pred, average="weighted")    
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
)


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    learning_rate=3e-5,  # Increased learning rate to combat underfitting
    weight_decay=0.01,      # Removed weight decay as it's for overfitting
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16=False,  # <-- set this to False
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_dir="./logs",
    lr_scheduler_type="linear",
    logging_steps=10,
    save_steps=10,
    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,
    optim="paged_adamw_32bit",
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=False,
    lr_scheduler_type="cosine",
    report_to="tensorboard",

)

# Now you can use:
trainer = SFTTrainer(
    model=finnbert_huggingface_model,
    args=training_args,
    train_dataset=ds_preprocessed["train"],
    eval_dataset=ds_preprocessed["validation"],
    peft_config=peft_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    packing=False,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    },

)

Map:   0%|          | 0/3054 [00:00<?, ? examples/s]

Map:   0%|          | 0/764 [00:00<?, ? examples/s]

/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_45711/4122420577.py:55: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# --- FOR QUICKER EVALUATION, USE A SUBSET OF THE VALIDATION DATA ---
# For example, use only the first 1000 samples for validation
val_df_subset = val_df.head(450) 

# Convert the subset to a Hugging Face Dataset
val_dataset_subset = Dataset.from_pandas(val_df_subset)

# Preprocess the subset
val_dataset_subset = val_dataset_subset.map(preprocess)

# Create a new trainer for evaluation on the subset (or just replace the eval_dataset)
# Note: To evaluate on the full dataset again, you would re-assign the original 'val_dataset'
trainer.eval_dataset = val_dataset_subset

# Now, this will run much faster
eval_results_subset = trainer.evaluate()

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

/Users/veikka/thesis-work/notebooks/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluation results on subset: {'eval_loss': 1.1310635805130005, 'eval_model_preparation_time': 0.0011, 'eval_accuracy': 0.35555555555555557, 'eval_precision': 0.3981470250094746, 'eval_recall': 0.35555555555555557, 'eval_f1': 0.3428113330303112, 'eval_runtime': 21.3936, 'eval_samples_per_second': 21.034, 'eval_steps_per_second': 1.356} 



In [20]:
print("Evaluation results on subset:")
for key, value in eval_results_subset.items():
    print(f"{key}: {value}")

Evaluation results on subset:
eval_loss: 1.1310635805130005
eval_model_preparation_time: 0.0011
eval_accuracy: 0.35555555555555557
eval_precision: 0.3981470250094746
eval_recall: 0.35555555555555557
eval_f1: 0.3428113330303112
eval_runtime: 21.3936
eval_samples_per_second: 21.034
eval_steps_per_second: 1.356


In [ ]:
trainer.save_model("../data/finbert_sentiment_model")

In [ ]:
model_sentiment_trained = AutoModelForSequenceClassification.from_pretrained("../data/finbert_sentiment_model")

In [ ]:
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=model_sentiment_trained,
    tokenizer=trainer.tokenizer,
    device="mps"
)

# Example usage:
result = sentiment_classifier("Tämä on todella hyvä uutinen!", return_all_scores=True)
print(result)

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Device set to use mps


[[{'label': 'LABEL_0', 'score': 0.18439872562885284}, {'label': 'LABEL_1', 'score': 0.48298409581184387}, {'label': 'LABEL_2', 'score': 0.3326171636581421}]]


/Users/veikka/thesis-work/notebooks/.venv/lib/python3.13/site-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
print(sentiment_classifier("Kyseessä voi olla loppuelämäsi nainen."))

[{'label': 'LABEL_0', 'score': 0.4107833206653595}]


In [ ]:
print(sentiment_classifier("ota koppi"))

[{'label': 'LABEL_2', 'score': 0.4013924300670624}]


In [ ]:
print(sentiment_classifier("vittu mitä paskaa."))

[{'label': 'LABEL_1', 'score': 0.46943631768226624}]


In [ ]:
print(sentiment_classifier("Kamppi, Kontula, Kluuvi"))

[{'label': 'LABEL_1', 'score': 0.48752835392951965}]


In [ ]:
print(sentiment_classifier("Tämä on todella vitun paska uutinen!"))

[{'label': 'LABEL_1', 'score': 0.5771677494049072}]


In [ ]:
(sentiment_classifier("Kuka tätä muka nyt ostaisi."))

[{'label': 'LABEL_1', 'score': 0.4029059410095215}]

In [ ]:
(sentiment_classifier("markkina näyttää voimakkaita hyytymisen merkkejä."))

[{'label': 'LABEL_2', 'score': 0.5822588205337524}]

In [ ]:
(sentiment_classifier("nokia julkaisi antoi tänään iltapäivällä positiivisen tulosvaroituksen."))

[{'label': 'LABEL_2', 'score': 0.3561813235282898}]